In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

In [ ]:
from pathlib import Path
import random
import time
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

import cv2
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchinfo import summary

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
PROCESSED_DIR = Path("../data/processed")
CHECKPOINT_DIR = Path("../outputs/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BATCH_SIZE = 16
EPOCHS = 30
LR = 1e-4
NUM_WORKERS = 0
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
class SRCNNDataset(Dataset):
    """
    Creates degraded RGB -> clean RGB pairs.

    Input:
        degraded target RGB image

    Target:
        original clean RGB image
    """

    def __init__(self, processed_dir, split):
        self.target_dir = Path(processed_dir) / split / "targets"
        self.target_files = sorted(self.target_dir.glob("*.npy"))

    def __len__(self):
        return len(self.target_files)

    def degrade_image(self, img):
        img = np.clip(img, 0, 1).astype(np.float32)

        # Gaussian blur
        blurred = cv2.GaussianBlur(img, (5, 5), sigmaX=1.0)

        h, w, _ = blurred.shape

        # Downsample then upsample: SRCNN-style degradation
        low_res = cv2.resize(
            blurred,
            (w // 2, h // 2),
            interpolation=cv2.INTER_CUBIC
        )

        restored_size = cv2.resize(
            low_res,
            (w, h),
            interpolation=cv2.INTER_CUBIC
        )

        noise = np.random.normal(0, 0.005, restored_size.shape).astype(np.float32)
        degraded = np.clip(restored_size + noise, 0, 1)

        return degraded.astype(np.float32)

    def __getitem__(self, idx):
        clean = np.load(self.target_files[idx]).astype(np.float32)
        degraded = self.degrade_image(clean)

        degraded = torch.from_numpy(degraded).permute(2, 0, 1)
        clean = torch.from_numpy(clean).permute(2, 0, 1)

        return degraded, clean

In [ ]:
train_dataset = SRCNNDataset(PROCESSED_DIR, "train")
val_dataset = SRCNNDataset(PROCESSED_DIR, "val")

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)

x, y = next(iter(train_loader))

print("Train samples:", len(train_dataset))
print("Val samples:", len(val_dataset))
print("Input:", x.shape)
print("Target:", y.shape)

In [ ]:
degraded, clean = train_dataset[np.random.randint(0, len(train_dataset))]

degraded_np = degraded.permute(1, 2, 0).numpy()
clean_np = clean.permute(1, 2, 0).numpy()

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.imshow(np.clip(degraded_np, 0, 1))
plt.title("Degraded Input")
plt.axis("off")

plt.subplot(1, 2, 2)
plt.imshow(np.clip(clean_np, 0, 1))
plt.title("Clean Target")
plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
class SRCNN(nn.Module):
    """
    SRCNN-style enhancement network.
    Input: degraded RGB
    Output: enhanced RGB
    """

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=9, padding=4),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 32, kernel_size=1),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 3, kernel_size=5, padding=2),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

In [ ]:
srcnn = SRCNN().to(device)

summary(
    srcnn,
    input_size=(BATCH_SIZE, 3, 128, 128),
    col_names=["input_size", "output_size", "num_params"],
)

In [ ]:
class SSIMLoss(nn.Module):
    def __init__(self, window_size=11):
        super().__init__()
        self.window_size = window_size

    def forward(self, pred, target):
        c1 = 0.01 ** 2
        c2 = 0.03 ** 2

        mu_x = F.avg_pool2d(pred, self.window_size, stride=1, padding=self.window_size // 2)
        mu_y = F.avg_pool2d(target, self.window_size, stride=1, padding=self.window_size // 2)

        sigma_x = F.avg_pool2d(pred * pred, self.window_size, stride=1, padding=self.window_size // 2) - mu_x ** 2
        sigma_y = F.avg_pool2d(target * target, self.window_size, stride=1, padding=self.window_size // 2) - mu_y ** 2
        sigma_xy = F.avg_pool2d(pred * target, self.window_size, stride=1, padding=self.window_size // 2) - mu_x * mu_y

        ssim = ((2 * mu_x * mu_y + c1) * (2 * sigma_xy + c2)) / (
            (mu_x ** 2 + mu_y ** 2 + c1) * (sigma_x + sigma_y + c2)
        )

        return 1.0 - ssim.mean()


class SRCNNHybridLoss(nn.Module):
    def __init__(self, mse_weight=0.8, ssim_weight=0.2):
        super().__init__()
        self.mse = nn.MSELoss()
        self.ssim = SSIMLoss()
        self.mse_weight = mse_weight
        self.ssim_weight = ssim_weight

    def forward(self, pred, target):
        mse_loss = self.mse(pred, target)
        ssim_loss = self.ssim(pred, target)

        total_loss = (
            self.mse_weight * mse_loss
            + self.ssim_weight * ssim_loss
        )

        return total_loss, mse_loss, ssim_loss


criterion = SRCNNHybridLoss(
    mse_weight=0.8,
    ssim_weight=0.2
)

optimizer = torch.optim.Adam(
    srcnn.parameters(),
    lr=LR
)

print("Training SRCNN with Hybrid Loss")
print("Loss = 0.8 MSE + 0.2 SSIM")

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    total_sum = 0.0
    mse_sum = 0.0
    ssim_sum = 0.0

    for degraded, clean in tqdm(loader, leave=False):
        degraded = degraded.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        optimizer.zero_grad()

        enhanced = model(degraded)

        total_loss, mse_loss, ssim_loss = criterion(enhanced, clean)

        total_loss.backward()
        optimizer.step()

        total_sum += total_loss.item()
        mse_sum += mse_loss.item()
        ssim_sum += ssim_loss.item()

    n = len(loader)

    return {
        "total": total_sum / n,
        "mse": mse_sum / n,
        "ssim": ssim_sum / n,
    }


@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()

    total_sum = 0.0
    mse_sum = 0.0
    ssim_sum = 0.0

    for degraded, clean in tqdm(loader, leave=False):
        degraded = degraded.to(device, non_blocking=True)
        clean = clean.to(device, non_blocking=True)

        enhanced = model(degraded)

        total_loss, mse_loss, ssim_loss = criterion(enhanced, clean)

        total_sum += total_loss.item()
        mse_sum += mse_loss.item()
        ssim_sum += ssim_loss.item()

    n = len(loader)

    return {
        "total": total_sum / n,
        "mse": mse_sum / n,
        "ssim": ssim_sum / n,
    }

In [ ]:
best_val_loss = float("inf")

history = {
    "train_total": [],
    "val_total": [],
    "train_mse": [],
    "val_mse": [],
    "train_ssim": [],
    "val_ssim": [],
}

start_time = time.time()

for epoch in range(EPOCHS):
    print(f"\nEpoch [{epoch + 1}/{EPOCHS}]")

    train_metrics = train_one_epoch(
        srcnn,
        train_loader,
        optimizer,
        criterion,
        device
    )

    val_metrics = validate(
        srcnn,
        val_loader,
        criterion,
        device
    )

    history["train_total"].append(train_metrics["total"])
    history["val_total"].append(val_metrics["total"])
    history["train_mse"].append(train_metrics["mse"])
    history["val_mse"].append(val_metrics["mse"])
    history["train_ssim"].append(train_metrics["ssim"])
    history["val_ssim"].append(val_metrics["ssim"])

    print(
        f"Train Total: {train_metrics['total']:.6f} | "
        f"MSE: {train_metrics['mse']:.6f} | "
        f"SSIM Loss: {train_metrics['ssim']:.6f}"
    )

    print(
        f"Val Total:   {val_metrics['total']:.6f} | "
        f"MSE: {val_metrics['mse']:.6f} | "
        f"SSIM Loss: {val_metrics['ssim']:.6f}"
    )

    if val_metrics["total"] < best_val_loss:
        best_val_loss = val_metrics["total"]

        checkpoint_path = CHECKPOINT_DIR / "best_srcnn_enhancement_hybrid.pth"

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": srcnn.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_loss": val_metrics["total"],
                "history": history,
                "model_name": "SRCNN_Enhancement_HybridLoss",
            },
            checkpoint_path
        )

print("✅ Best SRCNN Hybrid model saved")

elapsed = time.time() - start_time

print("\nSRCNN Training Complete")
print(f"Best Val Loss: {best_val_loss:.6f}")
print(f"Time: {elapsed / 60:.2f} minutes")

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(history["train_total"], label="Train Total", linewidth=2)
plt.plot(history["val_total"], label="Validation Total", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("Hybrid Loss")
plt.title("SRCNN Hybrid Loss Training Curve")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history["train_mse"], label="Train MSE", linewidth=2)
plt.plot(history["val_mse"], label="Validation MSE", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.title("SRCNN MSE Component")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
plt.plot(history["train_ssim"], label="Train SSIM Loss", linewidth=2)
plt.plot(history["val_ssim"], label="Validation SSIM Loss", linewidth=2)
plt.xlabel("Epoch")
plt.ylabel("SSIM Loss")
plt.title("SRCNN SSIM Component")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def show_srcnn_result(idx=None):
    if idx is None:
        idx = np.random.randint(0, len(val_dataset))

    degraded, clean = val_dataset[idx]

    srcnn.eval()

    enhanced = srcnn(
        degraded.unsqueeze(0).to(device)
    ).squeeze(0).cpu()

    degraded_np = degraded.permute(1, 2, 0).numpy()
    enhanced_np = enhanced.permute(1, 2, 0).numpy()
    clean_np = clean.permute(1, 2, 0).numpy()

    error = np.abs(enhanced_np - clean_np).mean(axis=-1)

    plt.figure(figsize=(18, 5))

    plt.subplot(1, 4, 1)
    plt.imshow(np.clip(degraded_np, 0, 1))
    plt.title("Degraded Input")
    plt.axis("off")

    plt.subplot(1, 4, 2)
    plt.imshow(np.clip(enhanced_np, 0, 1))
    plt.title("SRCNN Enhanced")
    plt.axis("off")

    plt.subplot(1, 4, 3)
    plt.imshow(np.clip(clean_np, 0, 1))
    plt.title("Clean Target")
    plt.axis("off")

    plt.subplot(1, 4, 4)
    plt.imshow(error, cmap="hot")
    plt.title("Error Map")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


for _ in range(5):
    show_srcnn_result()